In [ ]:
#### 01-intro

In [1]:
! uv add requests minsearch openai jupyter python-dotenv sentence-transformers

Resolved 166 packages in 6ms
Checked 144 packages in 56ms


In [2]:
#### 02-embeddings

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)
v1.shape

(384,)

In [7]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [8]:
v1.dot(dv)

np.float32(0.32332402)

In [9]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [10]:
v2.dot(dv)


np.float32(0.01973051)

In [11]:
#### Embedding Our Dataset

In [12]:
! wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-23 09:25:49--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 888 [text/plain]
Saving to: ‘ingest.py.2’

ingest.py.2         100%[===================>]     888  --.-KB/s    in 0s      

2026-06-23 09:25:49 (52.9 MB/s) - ‘ingest.py.2’ saved [888/888]



In [13]:
from ingest import load_faq_data

documents = load_faq_data()

In [14]:
len(documents)
documents[0]["question"]

'Course: When does the course start?'

In [16]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)
len(texts)
texts[0]


"Course: When does the course start? A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."

In [17]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [18]:
vectors[0]
vectors[0].shape

(384,)

In [19]:
import numpy as np
X = np.array(vectors)
type(X)
X

array([[-0.02670619, -0.12245757,  0.01594414, ..., -0.00230649,
        -0.11218392, -0.02365565],
       [-0.01099562, -0.11074749, -0.02536941, ...,  0.09022235,
        -0.02697364,  0.01975661],
       [-0.08896554, -0.06128183,  0.00775607, ...,  0.04059711,
         0.00479284, -0.02745946],
       ...,
       [-0.03652921,  0.01415428, -0.06838642, ...,  0.04316788,
         0.08105536, -0.02148629],
       [-0.13091598, -0.06990604, -0.00931881, ..., -0.00044338,
        -0.01285727,  0.01426919],
       [-0.07984785,  0.01926977,  0.02544985, ..., -0.0336803 ,
        -0.01884027,  0.05837052]], shape=(1350, 384), dtype=float32)

In [20]:
X.shape

(1350, 384)

In [21]:
#### Vector Search

In [22]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)
v_query.shape

(384,)

In [23]:
scores = X.dot(v_query)

In [24]:
scores  

array([ 0.48740596,  0.20991942,  0.7629412 , ..., -0.08637968,
        0.03759797, -0.03037032], shape=(1350,), dtype=float32)

In [25]:
# Best match
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.7629412))

In [26]:
documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [27]:
# Top 5 results
top5 = np.argsort(scores)[-5:]


In [28]:
top5

array([  7, 538, 907, 625,   2])

In [29]:
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [30]:
scores[top5]

array([0.7629412, 0.7579372, 0.7192135, 0.6536312, 0.5601001],
      dtype=float32)

In [31]:
top5 = np.argsort(-scores)[:5]

In [32]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.7629412
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579372
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192135
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions',

In [33]:
######## Vector Search with minsearch


In [34]:
# Creating the index
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [35]:
# Searching

In [36]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)
len(query_vector)
query_vector[0]
results = vindex.search(query_vector, num_results=5)

In [37]:
results[1]

{'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'The course has already started. Can I still join it?',
 'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
 'doc_id': '41aabbd7c5'}

In [38]:
# Filtering by course

In [39]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [40]:
results[1]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
 'doc_id': '69d122f12e'}

In [41]:
######## RAG with Vector Search

In [42]:
# Using RAGBase


In [43]:
! wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
! wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-23 09:27:33--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.1’

rag_helper.py.1     100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-23 09:27:34 (185 MB/s) - ‘rag_helper.py.1’ saved [2134/2134]

--2026-06-23 09:27:34--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200

In [44]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [45]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)


In [46]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [47]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, you can still join. If you want to receive a certificate, make sure you submit your project while submissions are still being accepted.'

In [48]:
query1 = "can i use olama"
assistant.rag(query1)

'Yes, you can use some local Ollama models. The course context says some local Ollama models support tool/function calling, but you may need to adapt the code because response formats and tool schemas can differ from OpenAI.'

In [49]:
## vector search

In [50]:

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [51]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [52]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still join even after the program has begun.\n\nIf you want to receive a certificate, make sure to submit your project while submissions are still open.'

In [53]:
vector_assistant.rag("can i use olama?")

'Yes — you can use Ollama. The course context says local models are allowed, and Ollama is supported. You may need to adapt the code because response formats and tool schemas can differ from the recommended provider.\n\nIf you meant “how do I install/use Ollama?”, I can help with that too.'

In [54]:
####### Vector Search with sqlitesearch

In [55]:
# Creating the index
! uv add sqlitesearch

Resolved 166 packages in 6ms
Checked 144 packages in 103ms


In [ ]:
### lsh (default): up to 100K vectors, random hyperplane projections
### ivf: 10K-500K vectors, K-means clustering
### hnsw: 10K-1M+ vectors, proximity graph (highest recall)

In [71]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [92]:
# Indexing the data
# vs_index.clear()
vs_index.fit(vectors, documents)

In [97]:
vs_index.search(model.encode("linux"), num_results=1)

[{'course': 'data-engineering-zoomcamp',
  'section': 'Environment & Setup',
  'question': 'Which operating systems does the course support?',
  'answer': 'Linux, macOS, and Windows all work. Students in the most recent cohorts have completed the course on all three. Linux is the smoothest by default.\n\nOn Windows, install WSL2 and run everything inside a WSL distro from the start. Git Bash and MINGW64 are not always sufficient for the shell scripts used in later modules.',
  'doc_id': 'a0e992e6ef'}]

In [98]:
# Search

In [99]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [101]:
results[4]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I get support if I take the course in the self-paced mode?',
 'answer': 'Yes, the Slack channel remains open and you can ask questions there. However, always search the channel first and check the FAQ, as most likely your questions are already answered here.\n\nYou can also tag the bot `@ZoomcampQABot` to help you conduct the search, but don’t rely on its answers 100%.',
 'doc_id': 'c207b8614e'}

In [102]:
# Filtering by course

In [103]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [104]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere

In [105]:
vs_index.close()

In [ ]:
###!!! Continue in vector_persistant